# Quickstart: aerial imaging, focus and the process window

`highuvlith` simulates optical lithography from VUV (157 nm) through EUV (13.5 nm)
and BEUV (6.7 nm) to soft and hard X-rays. This notebook walks the core imaging
pipeline with the F$_2$ excimer laser at 157.63 nm and a refractive projector at
NA 0.75:

1. the one-line `simulate_line_space` helper;
2. the explicit `SimulationEngine` with a commensurate grid;
3. through-focus imaging with the exact defocus model;
4. a dose-aware focus-exposure (ED) process window;
5. a 2D resist profile.

Every result object carries a `.status` badge from the
[capability matrix](../../docs/capability-matrix.md):
✅ implemented and validated, 🔶 simplified (stated approximations), 🧪 theoretical.

Run it from a checkout after `maturin develop --release` (highuvlith is not on PyPI).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import viz

%matplotlib inline
print("highuvlith", huv.__version__)

## Physics context

The aerial image is the light intensity the mask projects into the wafer plane.
`highuvlith` computes it with the **Hopkins partially coherent** model: the
transmission cross-coefficient (TCC) of source and pupil is factorized into a few
coherent **SOCS** kernels, and the image is the incoherent sum of the mask
filtered by each kernel. Images are **normalized to the clear field** of the same
illumination and optics, so 1.0 means "as bright as an open frame".

Two definitions used throughout:

- image contrast $C = (I_{max} - I_{min}) / (I_{max} + I_{min})$;
- NILS $= w \cdot \lvert dI/dx \rvert / I_{th}$, the normalized image log-slope at the
  printing threshold $I_{th}$ (larger means a sharper edge; about 2 is a common rule of
  thumb for a manufacturable image).

## 1. The one-liner

`simulate_line_space` builds the source, optics, a periodic line/space mask and a
grid, then returns the image and its metrics. Here: **100 nm opaque lines on a
300 nm pitch** at NA 0.75 with a conventional σ = 0.7 fill.

The helper always picks a **commensurate** grid: the simulated field must hold a
whole number of pitches, because the FFT treats the field as periodic. The
requested 2 nm pixel is adjusted so that 128 pixels span exactly one 300 nm pitch;
`config` records what was actually used.

In [ ]:
r = huv.simulate_line_space(100.0, 300.0, grid_size=128, pixel_nm=2.0)
print(f"contrast {r.contrast:.3f}   printed CD {r.cd_nm:.1f} nm   NILS {r.nils:.2f}   status {r.status}")
print("pixel used:", r.config["pixel_nm"], "nm (requested", r.config["pixel_nm_requested"], "nm),",
      r.config["periods_in_field"], "period(s) in a", r.config["field_nm"], "nm field")

The printed CD is measured at a 0.3 intensity threshold, so it sits a little below
the 100 nm drawn line: the image of a line is blurred by the finite pupil.

## 2. Full control: the `SimulationEngine`

For anything beyond the defaults you assemble the configuration objects yourself.
`mask.commensurate_grid(...)` gives the right grid for a periodic mask (building an
engine on an incommensurate grid raises a `UserWarning`). We use a two-pitch field
so the image shows more than one line. A larger field samples the TCC more
finely, so it needs more SOCS kernels for the same accuracy: always check the
captured energy fraction (with 20 kernels this field would capture only about
96 % and the CD would read about 1 nm wide).

In [ ]:
source = huv.SourceConfig.f2_laser(sigma=0.7)          # 157.63 nm, conventional fill
optics = huv.OpticsConfig(numerical_aperture=0.75)     # dry refractive projector
mask = huv.MaskConfig.line_space(cd_nm=100.0, pitch_nm=300.0)
grid = huv.GridConfig(size=128, pixel_nm=600.0 / 128)   # 600 nm field = exactly 2 pitches
engine = huv.SimulationEngine(source, optics, mask, grid=grid, max_kernels=80)

aerial = engine.compute_aerial_image(focus_nm=0.0)
print("field", grid.field_size_nm(), "nm, periods", grid.periods_in_field(300.0),
      "commensurate:", grid.is_commensurate_with(300.0))
print(f"contrast {aerial.image_contrast():.3f}, CD {aerial.cd(threshold=0.3, tone='dark'):.2f} nm")
d = engine.imaging_diagnostics()
print(f"{d['num_kernels']} SOCS kernels capture {d['captured_energy_fraction']:.4f} of the TCC trace")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
viz.plot_aerial(aerial, ax=ax0, title="Aerial image (F2, 157.63 nm, NA 0.75)")
viz.plot_cross_section(aerial, threshold=0.3, ax=ax1, title="Cross-section at y = 0")
fig.tight_layout()

## 3. Through focus

Defocus adds a phase to every pupil ray. The engine evaluates it **exactly** at
every source point, $\phi = (2 \pi n / \lambda) z (1 - \sqrt{1 - (\mathrm{NA} \rho / n)^2})$,
inside the pupil (an older per-frequency approximation is still available as
`defocus_model="kernel_phase"`; notebook 04 shows why it is wrong for
partially coherent light). `compute_through_focus` reuses one mask spectrum for
all planes. For an aberration-free system the contrast is symmetric about best
focus.

In [ ]:
focuses = np.linspace(-400.0, 400.0, 17)
planes = engine.compute_through_focus(list(focuses))
contrast = np.array([p.image_contrast() for p in planes])

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
ax0.plot(focuses, contrast, "o-")
ax0.set(xlabel="defocus (nm)", ylabel="image contrast", title="Contrast through focus", ylim=(0, 1))
ax0.grid(alpha=0.3)
for z in (0.0, 200.0, 400.0):
    p = planes[int(np.argmin(abs(focuses - z)))]
    x, cut = p.cross_section(0.0)
    ax1.plot(x, cut, label=f"{z:+.0f} nm")
ax1.axhline(0.3, ls="--", lw=1, color="grey")
ax1.set(xlabel="x (nm)", ylabel="relative intensity", title="Cross-sections")
ax1.legend(title="defocus")
fig.tight_layout()
print("symmetric about focus:", np.allclose(contrast, contrast[::-1], atol=1e-9))

## 4. A dose-aware process window

The process window asks: over which range of **dose and focus** does the printed
CD stay within ±10 % of target? `huv.process_window` uses a constant-threshold
resist (🔶): a point clears where dose × intensity ≥ $E_{th}$. Without an explicit
$E_{th}$ it is chosen so that the line prints at its drawn CD in focus at
30 mJ/cm². Each dose row of the focus-exposure matrix is therefore different,
which is what makes exposure latitude (EL) meaningful.

- **Bossung curves**: CD vs focus, one curve per dose;
- **ED window**: the dose range in spec at each focus, with the largest inscribed
  rectangle;
- **EL vs DOF**: the trade-off curve; "DOF at 5 % EL" is a common single-number
  summary.

In [ ]:
pw = huv.process_window(
    100.0, 300.0, grid_size=64, pixel_nm=4.0,
    doses_mj_cm2=list(np.linspace(24.0, 36.0, 13)),
    focuses_nm=list(np.linspace(-400.0, 400.0, 17)),
)
w5 = pw.dof_at_el(5.0)
print(f"E_th = {pw.dose_to_clear_mj_cm2:.2f} mJ/cm2, dose-to-size = {pw.dose_to_size():.2f} mJ/cm2")
print(f"DOF at 5 % EL = {w5['dof_nm']:.0f} nm, status {pw.status}")
for note in pw.notes:
    print("  note:", note)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
viz.plot_bossung(pw, ax=axes[0])
viz.plot_ed_window(pw, ax=axes[1])
viz.plot_el_vs_dof(pw, ax=axes[2])
fig.tight_layout()

## 5. A 2D resist profile (🔶)

With a `ResistConfig` the engine also runs a Dill exposure, a Gaussian
post-exposure bake and Mack development. This 2D path is simplified: the exposure
is depth-averaged and development proceeds vertically along the centre row only.
For z-resolved exposure, sidewalls and level-set development see notebooks 03 and
06.

In [ ]:
resist_engine = huv.SimulationEngine(
    source, optics, mask, resist=huv.ResistConfig.vuv_fluoropolymer(), grid=grid, max_kernels=80
)
fig, ax = plt.subplots(figsize=(7, 3.5))
for dose in (20.0, 30.0, 45.0):
    prof = resist_engine.compute_resist_profile(dose_mj_cm2=dose, focus_nm=0.0, dev_time_s=60.0)
    ax.plot(prof.x_nm, prof.height_nm, label=f"{dose:.0f} mJ/cm²")
ax.set(xlabel="x (nm)", ylabel="remaining resist (nm)",
       title="Developed profile after 60 s")
ax.legend()
ax.grid(alpha=0.3)
print("status", prof.status, "-", "; ".join(prof.notes))

## Where to go next

| Notebook | Topic |
|---|---|
| `02_source_gallery.ipynb` | all fourteen source families through one pipeline |
| `03_deep_litho.ipynb` | LIGA, interference and volumetric (z-resolved) exposure |
| `04_imaging_physics.ipynb` | partial coherence, off-axis DOF, vector TE/TM, immersion, per-λ imaging |
| `05_source_physics.ipynb` | source landscape, derived quantities, throughput, honest gaps |
| `06_process_and_optimization.ipynb` | ED windows, ILT/OPC/SRAF, level set and CAR, Talbot/EUV-IL |

Docs: [Python API](../../docs/python-api.md),
[Masks and metrics](../../docs/masks-and-metrics.md),
[Imaging pipeline](../../docs/pipeline.md).